# Aufgabe 1

Implementieren Sie die LU -Zerlegung für Tridiagonal-Matrizen aus Aufgabe (3)(a) in Python und testen Sie Ihre Funktion an der Tridiagonalmatrix mit $bi = ci+1 = −i$ für $i = 1 ...,n −1$ und $ai = 4$
für alle $i = 1,...,n, n = 8$

In [1]:
import numpy as np 
import matplotlib.pyplot as plt 

/home/lme/.local/lib/python3.9/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# Make the Tridiagonal Matrix for Testing

def create_tridiagonal_matrix(n):
    a = 4 * np.ones(n)
    b = -np.arange(1, n)
    c = -np.arange(1, n)
    
    matrix = np.zeros((n, n))
    
    for i in range(n):
        # a on the diagonal
        matrix[i, i] = a[i]
        if i > 0:
            # b on the lower diagonal
            matrix[i, i-1] = b[i-1]
        if i < n-1:
            # c on the upper diagonal
            matrix[i, i+1] = c[i]
    
    return matrix

n = 8
T = create_tridiagonal_matrix(n)
print(T)


[[ 4. -1.  0.  0.  0.  0.  0.  0.]
 [-1.  4. -2.  0.  0.  0.  0.  0.]
 [ 0. -2.  4. -3.  0.  0.  0.  0.]
 [ 0.  0. -3.  4. -4.  0.  0.  0.]
 [ 0.  0.  0. -4.  4. -5.  0.  0.]
 [ 0.  0.  0.  0. -5.  4. -6.  0.]
 [ 0.  0.  0.  0.  0. -6.  4. -7.]
 [ 0.  0.  0.  0.  0.  0. -7.  4.]]


Goal is to calculate the matrix equation $Ax = b$. For this we decompose A into

$PA = LU$ where $L$ is a lower triangular matrix and $U$ is a upper triagonal matrix, $P$ is the permutation matrix

In [6]:
# First test the decomposition using the respective scipy method
import scipy
import pprint
import scipy.linalg

P,L,U = scipy.linalg.lu(T)

pprint.pprint(P)
pprint.pprint(L)
pprint.pprint(U)

array([[1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0.]])
array([[ 1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ],
       [-0.25      ,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -0.        , -0.        ,  1.        ,
         0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , -0.        , -0.        , -0.   

In [14]:
# Make a Function for Matrix Multiplikation

def mult_matrix(M,N):
    """
    Multiplies square matrices of dimension M and N
    """
    return np.dot(M,N)

# And generate the pivoting matrix

def pivot_matrix(M):
    m = len(M)
    # Unse np.eye to generate a diaognal array with all ones
    id_mat = np.eye(m)
    for j in range(m):
        row = np.argmax(np.abs(M[j:,j])) + j
        if j != row:
            # Swap the rows
            id_mat[[j,row]] = id_mat[[row,j]]
    return id_mat

print(pivot_matrix(T))

# Calulate the initial pivot matrix * T

print(mult_matrix(pivot_matrix(T),T))

[[1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0.]]
[[ 4. -1.  0.  0.  0.  0.  0.  0.]
 [-1.  4. -2.  0.  0.  0.  0.  0.]
 [ 0. -2.  4. -3.  0.  0.  0.  0.]
 [ 0.  0. -3.  4. -4.  0.  0.  0.]
 [ 0.  0.  0.  0. -5.  4. -6.  0.]
 [ 0.  0.  0.  0.  0. -6.  4. -7.]
 [ 0.  0.  0.  0.  0.  0. -7.  4.]
 [ 0.  0.  0. -4.  4. -5.  0.  0.]]


I used the partial pivoting algorithm:

1. Select the Largest entry in a column and use it as a pivot element
2. To do this step we need to multiply $P_1A$, where $P_1$ is the permutation matrix
3. Then we do the elimination $

In [23]:
def lu_decomposition(A):
    """Performs an LU Decomposition of A (which must be square)
    into PA = LU. The function returns P, L and U."""
    n = len(A)
    L = np.zeros((n, n))
    U = np.zeros((n, n))
    P = pivot_matrix(A)
    PA = mult_matrix(P, A)
    print(PA)
    for j in range(n):
        L[j, j] = 1.0
        for i in range(j+1):
            s1 = sum(U[k, j] * L[i, k] for k in range(i))
            U[i, j] = PA[i, j] - s1
        for i in range(j, n):
            s2 = sum(U[k, j] * L[i, k] for k in range(j))
            L[i, j] = (PA[i, j] - s2) / U[j, j]

    return P, L, U

P,L,U = lu_decomposition(T)

print(P)
print(L)
print(U)

[[ 4. -1.  0.  0.  0.  0.  0.  0.]
 [-1.  4. -2.  0.  0.  0.  0.  0.]
 [ 0. -2.  4. -3.  0.  0.  0.  0.]
 [ 0.  0. -3.  4. -4.  0.  0.  0.]
 [ 0.  0.  0.  0. -5.  4. -6.  0.]
 [ 0.  0.  0.  0.  0. -6.  4. -7.]
 [ 0.  0.  0.  0.  0.  0. -7.  4.]
 [ 0.  0.  0. -4.  4. -5.  0.  0.]]
[[1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0.]]
[[ 1.          0.          0.          0.          0.          0.
   0.          0.        ]
 [-0.25        1.          0.          0.          0.          0.
   0.          0.        ]
 [ 0.         -0.53333333  1.          0.          0.          0.
   0.          0.        ]
 [ 0.          0.         -1.02272727  1.          0.          0.
   0.          0.        ]
 [ 0.          0.          0.          0.          1.          0.
   0.          0.        ]
 [ 0.          0.          0.        